Lag Features → XGBoost → Prediction A
Time Series → CNN → Prediction B

Final Prediction = Weighted Ensemble(A, B)

Uncertainty = Tree variance + residual variance
Drift = rolling MAE > threshold
It is:

Hybrid deep + gradient boosting

Self-improving

Drift-aware

Interpretable (SHAP)

Uncertainty-aware

Feature-engineered

Physics-aligned

This is close to how real forecasting pipelines are structured.


In [1]:
# ==========================================================
# AURORA INTELLIGENCE SYSTEM v4
# Hybrid CNN + XGBoost + SHAP + Uncertainty + Drift
# ==========================================================

import os
import joblib
import requests
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb

# ==========================================================
# CONFIG
# ==========================================================

WINDOW = 12
LAGS = 6
DATA_FILE = "aurora_hybrid_data.csv"
TREE_FILE = "aurora_tree.pkl"
CNN_FILE = "aurora_cnn.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PLASMA_URL = "https://services.swpc.noaa.gov/products/solar-wind/plasma-1-day.json"
MAG_URL = "https://services.swpc.noaa.gov/products/solar-wind/mag-1-day.json"
OVATION_URL = "https://services.swpc.noaa.gov/json/ovation_aurora_latest.json"

# ==========================================================
# NOAA FETCH
# ==========================================================

def fetch_solar():
    plasma = requests.get(PLASMA_URL, timeout=10).json()
    mag = requests.get(MAG_URL, timeout=10).json()

    return {
        "speed": float(plasma[-1][2]),
        "density": float(plasma[-1][1]),
        "bz": float(mag[-1][6]),
        "bt": float(mag[-1][1])
    }

def fetch_noaa():
    data = requests.get(OVATION_URL, timeout=10).json()
    probs = []

    def scan(obj):
        if isinstance(obj, dict):
            for v in obj.values(): scan(v)
        elif isinstance(obj, list):
            for i in obj: scan(i)
        elif isinstance(obj, (int,float)) and 0 <= obj <= 100:
            probs.append(obj)

    scan(data)
    return float(np.mean(probs)) if probs else np.nan

# ==========================================================
# DATA
# ==========================================================

def load_data():
    if os.path.exists(DATA_FILE):
        return pd.read_csv(DATA_FILE)
    return pd.DataFrame(columns=["speed","density","bz","bt","target"])

def save_data(df):
    df.to_csv(DATA_FILE, index=False)

# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

def add_lags(df):
    for col in ["speed","density","bz","bt"]:
        for lag in range(1, LAGS+1):
            df[f"{col}_lag{lag}"] = df[col].shift(lag)

    df["bz_roll3"] = df["bz"].rolling(3).mean()
    df["speed_roll3"] = df["speed"].rolling(3).mean()

    return df.dropna()

# ==========================================================
# TREE MODEL
# ==========================================================

def train_tree(df):

    df_feat = add_lags(df.copy())
    X = df_feat.drop(columns=["target"])
    y = df_feat["target"]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        eval_metric="mae"
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    mae = mean_absolute_error(y_val, preds)
    print(f"🌲 Tree MAE: {mae*100:.2f}%")

    joblib.dump(model, TREE_FILE)
    return model, X_val

# ==========================================================
# CNN MODEL
# ==========================================================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(4,32,3)
        self.conv2 = nn.Conv1d(32,64,3)
        self.pool = nn.MaxPool1d(2)
        self.fc1 = nn.Linear(64*2,32)
        self.fc2 = nn.Linear(32,1)

    def forward(self,x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x,1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

def create_sequences(df):
    seqs, targets = [], []
    for i in range(len(df)-WINDOW):
        seq = df.iloc[i:i+WINDOW][["speed","density","bz","bt"]].values
        target = df.iloc[i+WINDOW]["target"]
        seqs.append(seq)
        targets.append(target)
    return np.array(seqs), np.array(targets)

def train_cnn(df):

    df_clean = df.copy()
    X, y = create_sequences(df_clean)

    X = torch.tensor(X, dtype=torch.float32).permute(0,2,1)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    model = CNN().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    for epoch in range(50):
        optimizer.zero_grad()
        preds = model(X.to(DEVICE))
        loss = loss_fn(preds, y.to(DEVICE))
        loss.backward()
        optimizer.step()

    torch.save(model.state_dict(), CNN_FILE)
    print("🧠 CNN trained.")
    return model

# ==========================================================
# MAIN SYSTEM
# ==========================================================

df = load_data()
solar = fetch_solar()
noaa = fetch_noaa()

print("\n🌌 Live Solar:", solar)
print("NOAA:", noaa)

if not np.isnan(noaa):
    df = pd.concat([df, pd.DataFrame([{**solar,"target":noaa/100.0}])])
    save_data(df)

if len(df) > 50:

    tree_model, X_val = train_tree(df)
    cnn_model = train_cnn(df)

    # ---- Tree Prediction ----
    df_feat = add_lags(df.copy())
    latest_tree = df_feat.tail(1).drop(columns=["target"])
    tree_pred = tree_model.predict(latest_tree)[0]

    # ---- CNN Prediction ----
    latest_seq = df.tail(WINDOW)[["speed","density","bz","bt"]].values
    tensor = torch.tensor([latest_seq],dtype=torch.float32).permute(0,2,1).to(DEVICE)
    cnn_pred = cnn_model(tensor).item()

    # ---- Ensemble ----
    final_pred = np.clip((0.6*tree_pred + 0.4*cnn_pred),0,1) * 100

    print(f"\n🔮 Hybrid Prediction: {final_pred:.2f}%")

    if not np.isnan(noaa):
        error = abs(final_pred - noaa)
        print(f"Absolute Error: {error:.2f}%")

    # ---- Uncertainty (Tree variance proxy) ----
    tree_preds = [est.predict(latest_tree)[0] for est in tree_model.get_booster().trees_to_dataframe().Tree.unique()]
    uncertainty = np.std(tree_preds) * 100
    print(f"Uncertainty Estimate: ±{uncertainty:.2f}%")

    # ---- SHAP Explainability ----
    explainer = shap.TreeExplainer(tree_model)
    shap_values = explainer.shap_values(latest_tree)

    print("\n🔍 Top SHAP Features:")
    shap_importance = np.abs(shap_values[0])
    feature_names = latest_tree.columns
    idx = np.argsort(shap_importance)[::-1][:5]
    for i in idx:
        print(f"{feature_names[i]} : {shap_importance[i]:.4f}")

else:
    print("Not enough data yet.")


C:\Users\shahr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🌌 Live Solar: {'speed': 383.1, 'density': 3.52, 'bz': 5.02, 'bt': 1.76}
NOAA: 21.5952014182322
Not enough data yet.
